In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("teaching_prices_messy.csv")

df.head()

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [9]:
clean = pd.read_csv("teaching_prices_cleaned.csv")



In [11]:
summary = clean.groupby("item")["price"].agg(
    Mean="mean",
    Median="median",
    Mode=lambda x: list(x.mode())
)

summary

,Mean,Median,Mode
item,,,
Oil B,324.833333,324.00,"[304.0, 310.0, 317.0, 324.0, 330.0, 337.0, 344.0]"
Pulse C,205.916667,206.00,"[202.5, 208.5]"
Staple A,111.282609,112.50,"[107.0, 112.5, 114.5]"
Sweetener D,166.750000,166.75,"[149.0, 153.0, 153.5, 155.0, 157.5, 159.0, 159..."


In [12]:
clean["date"] = pd.to_datetime(clean["date"])

In [13]:
daily_prices = clean.groupby(
    ["item", "date"]
)["price"].mean().reset_index()

daily_prices.head()

,item,date,price
0,Oil B,2026-01-01,301.0
1,Oil B,2026-01-08,307.0
2,Oil B,2026-01-15,313.0
3,Oil B,2026-01-22,314.0
4,Oil B,2026-01-29,320.0


In [14]:
first_last = daily_prices.sort_values(
    "date"
).groupby("item").agg(
    First_Price=("price", "first"),
    Last_Price=("price", "last")
)

first_last

,First_Price,Last_Price
item,,
Oil B,301.0,347.0
Pulse C,203.0,210.5
Staple A,104.0,117.5
Sweetener D,152.0,181.5


In [15]:
first_last["Percentage_Change"] = (
    (first_last["Last_Price"] - first_last["First_Price"])
    / first_last["First_Price"]
) * 100

first_last.round(2)

,First_Price,Last_Price,Percentage_Change
item,,,
Oil B,301.0,347.0,15.28
Pulse C,203.0,210.5,3.69
Staple A,104.0,117.5,12.98
Sweetener D,152.0,181.5,19.41


In [16]:
clean["flagged"] = clean["price"] > 1000

clean[clean["flagged"]]

,date,item,city,price,unit,data_label,flagged


In [18]:
for item, flagged_price in [
    ("Staple A", 1250),
    ("Sweetener D", 9999)
]:

    values = clean.loc[
        clean["item"] == item, "price"
    ].dropna()

    without_flag = values[
        values != flagged_price
    ]

    print("\nItem:", item)

    print("Mean with flag:", values.mean())
    print("Median with flag:", values.median())

    print("Mean without flag:", without_flag.mean())
    print("Median without flag:", without_flag.median())


Item: Staple A
Mean with flag: 111.28260869565217
Median with flag: 112.5
Mean without flag: 111.28260869565217
Median without flag: 112.5

Item: Sweetener D
Mean with flag: 166.75
Median with flag: 166.75
Mean without flag: 166.75
Median without flag: 166.75


In [19]:
final_summary = summary.join(first_last)

final_summary["Percentage_Change"] = (
    first_last["Percentage_Change"]
)

final_summary.round(2)

,Mean,Median,Mode,First_Price,Last_Price,Percentage_Change
item,,,,,,
Oil B,324.83,324.00,"[304.0, 310.0, 317.0, 324.0, 330.0, 337.0, 344.0]",301.0,347.0,15.28
Pulse C,205.92,206.00,"[202.5, 208.5]",203.0,210.5,3.69
Staple A,111.28,112.50,"[107.0, 112.5, 114.5]",104.0,117.5,12.98
Sweetener D,166.75,166.75,"[149.0, 153.0, 153.5, 155.0, 157.5, 159.0, 159...",152.0,181.5,19.41
